In [ ]:
# Dependencies
!pip install pandas numpy pmdarima matplotlib statsmodels scipy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from pmdarima import auto_arima
import matplotlib.pyplot as plt
import warnings
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')

# Loading the data
df = pd.read_excel('/content/data.xlsx')

# Converting date column to DateTime format
df["Date of sample collection"] = pd.to_datetime(df["Date of sample collection"],
                                                dayfirst=True,
                                                errors='coerce')

# Filtering for Enterococcus
enterococcus_df = df[df['Organism'].str.lower().str.contains('enterococcus', na=False)]

# Cleaning/filtering remarks
enterococcus_df = enterococcus_df[enterococcus_df['Remarks'].isin(['resistant', 'susceptible'])]

# Creating monthly time series
enterococcus_df['Month'] = enterococcus_df['Date of sample collection'].dt.to_period('M')

# Calculating resistance rates
antibiotic_resistance = (enterococcus_df.groupby(['Month', 'Antibiotic'])
                         .apply(lambda x: pd.Series({
                             'resistance_rate': (x['Remarks'] == 'resistant').mean(),
                             'total_samples': x.shape[0]
                         }))
                         .reset_index())

# Creating pivot-table
pivot_df = antibiotic_resistance.pivot_table(index='Month',
                                           columns='Antibiotic',
                                           values='resistance_rate')

# Converting Period index to DateTime
pivot_df.index = pivot_df.index.to_timestamp()

# Function to check if seasonality exists
def has_seasonality(series, m=12):
    """Check if seasonal differencing is needed using ADF test."""
    if len(series) < 2 * m:
        return False  # not enough data
    try:
        p_value = adfuller(series.dropna().diff(m).dropna())[1]
        return p_value < 0.05  # p-value is small, seasonality is possible
    except:
        return False  # assume no seasonality

# Function to run time series analysis
def analyze_antibiotic(series, antibiotic_name):
    if series.dropna().empty or len(series.dropna()) < 24:
        print(f"Skipping {antibiotic_name} - only {len(series.dropna())} months of data available")
        return

    # Auto ARIMA (Non-Seasonal)
    arima_model = auto_arima(series,
                            seasonal=False,
                            stepwise=True,
                            suppress_warnings=True,
                            error_action='ignore')

    # Seasonal ARIMA (SARIMA) only if seasonality exists
    is_seasonal = has_seasonality(series, m=12)

    sarima_model = None
    if is_seasonal:
        sarima_model = auto_arima(series,
                                  seasonal=True,
                                  m=12,
                                  stepwise=True,
                                  suppress_warnings=True,
                                  error_action='ignore',
                                  seasonal_test="ch",  # Use CH Test instead of OCSB
                                  max_order=5)  # Reduce complexity

    # Forecast
    forecast_steps = 6
    arima_forecast = arima_model.predict(n_periods=forecast_steps)
    sarima_forecast = None
    if sarima_model:
        sarima_forecast = sarima_model.predict(n_periods=forecast_steps)

    # Plot results
    plt.figure(figsize=(12, 6))
    plt.plot(series, label='Observed')
    plt.plot(series.index[-forecast_steps:], arima_forecast, label='ARIMA Forecast', linestyle="dashed")
    if sarima_forecast is not None:
        plt.plot(series.index[-forecast_steps:], sarima_forecast, label='SARIMA Forecast', linestyle="dotted")
    plt.title(f'Resistance Rate Forecast for {antibiotic_name}')
    plt.xlabel('Date')
    plt.ylabel('Resistance Rate')
    plt.legend()
    plt.show()

    # Print model summaries
    print(f"\n{antibiotic_name} ARIMA Order: {arima_model.order}")
    if sarima_model:
        print(f"{antibiotic_name} SARIMA Order: {sarima_model.order} Seasonal Order: {sarima_model.seasonal_order}\n")
    else:
        print(f"{antibiotic_name} SARIMA skipped (no detected seasonality)\n")


# Analyze each antibiotic with sufficient data
for antibiotic in pivot_df.columns:
    ts = pivot_df[antibiotic].dropna().asfreq('MS').ffill()
    analyze_antibiotic(ts, antibiotic)


